<a href="https://colab.research.google.com/github/ghroyd1110/Git-Commands/blob/Test/GeneratorExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf

In [ ]:
!pip install pypdf

In [ ]:
!pip install docling

In [ ]:
import os
import re
import json
import pandas as pd
from docling.document_converter import DocumentConverter
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. CONFIGURATION
ROOT_DIR = '/content/drive/MyDrive/Permits'
JSON_OUTPUT = '/content/drive/MyDrive/PermitsOutput/Master_Standardized_Inventory.json'

def process_single_pdf(pdf_path, converter):
    """Analyzes a PDF and extracts unique standardized generator records."""
    try:
        result = converter.convert(pdf_path)
        doc = result.document
        md_text = doc.export_to_markdown()

        # --- METADATA EXTRACTION ---
        def get_meta(label):
            pattern = f"{label}:?\\s*\\*?\\*?([^\\n|]+)"
            match = re.search(pattern, md_text, re.IGNORECASE)
            if match:
                clean_val = match.group(1).strip().replace('*', '')
                return clean_val.split('  ')[0].strip()
            return "N/A"

        permit_meta = {
            "permittee": get_meta("PERMITTEE"),
            "facility": get_meta("FACILITY"),
            "permit_no": get_meta("PERMIT No."),
            "place_id": get_meta("PLACE ID"),
            "source_file": os.path.basename(pdf_path)
        }

        inventory = []
        seen_in_this_permit = set() # Local set to deduplicate within one file

        # --- TABLE EXTRACTION ---
        for item, _ in doc.iterate_items():
            if item.__class__.__name__ == "TableItem":
                df = item.export_to_dataframe()
                df.columns = [str(c).strip().replace('\n', ' ').upper() for c in df.columns]
                rows = df.to_dict(orient='records')

                for row in rows:
                    row_str = str(row).upper()
                    raw_equip_type = str(row.get("EQUIPMENT TYPE", row.get("EQUIPMENT", "UNKNOWN")))

                    # 1. FILTER: Is it a generator?
                    is_gen = any(x in raw_equip_type.upper() or x in row_str
                                 for x in ["GEN", "ENGINE", "INTERNAL COMBUSTION", "GENSET"])

                    if not is_gen:
                        continue

                    # 2. SMART CAPACITY DETECTION
                    capacity = "N/A"
                    for col_name, value in row.items():
                        col_upper = str(col_name).upper()
                        if any(k in col_upper for k in ["CAPACITY", "RATING", "HP", "KW", "MW", "SIZE"]):
                            if pd.notna(value) and str(value).strip() != "":
                                val_str = str(value).strip()
                                has_unit = any(u in val_str.upper() for u in ["HP", "KW", "MW", "BHP"])
                                if not has_unit:
                                    if "HP" in col_upper or "BHP" in col_upper: val_str += " hp"
                                    elif "KW" in col_upper: val_str += " kW"
                                    elif "MW" in col_upper: val_str += " MW"
                                capacity = val_str
                                break

                    # 3. DEDUPLICATION LOGIC
                    # Create a fingerprint: "94799_250 HP_DETROIT DIESEL"
                    # This prevents the same unit from appearing multiple times
                    fingerprint = f"{permit_meta['permit_no']}_{capacity}_{raw_equip_type}".strip().upper()

                    if fingerprint in seen_in_this_permit:
                        continue
                    seen_in_this_permit.add(fingerprint)

                    # 4. REMAINING MAPPING
                    fuel = "Diesel" if any(x in row_str for x in ["DIESEL", "FUEL OIL", "DISTILLATE"]) \
                           else "Natural Gas" if "GAS" in row_str \
                           else "Propane" if "PROPANE" in row_str \
                           else "TBD"

                    status = "Retired" if "RETIRED" in row_str else "Active"
                    purpose = "Emergency Backup" if "EMERGENCY" in row_str \
                              else "Peaking" if "PEAK" in row_str \
                              else "Continuous"

                    inventory.append({
                        "metadata": permit_meta,
                        "specs_standardized": {
                            "equipment_type": raw_equip_type,
                            "is_generator": is_gen,
                            "generator_location_bldg": row.get("SITE AREA OR BUILDING", "N/A"),
                            "nameplate_capacity": capacity,
                            "fuel_type": fuel,
                            "operational_status": status,
                            "primary_purpose": purpose
                        }
                    })
        return inventory
    except Exception as e:
        print(f"   ! Error processing {os.path.basename(pdf_path)}: {e}")
        return []

# 3. MAIN CRAWLER LOOP
converter = DocumentConverter()
processed_count = 0
found_generators = 0

with open(JSON_OUTPUT, 'a', encoding='utf-8') as out_file:
    print(f"Scanning {ROOT_DIR} for 'Final' permits...")

    for root, dirs, files in os.walk(ROOT_DIR):
        is_final_folder = "FINAL" in root.upper()
        for file in files:
            if file.lower().endswith(".pdf"):
                if is_final_folder or "FINAL" in file.upper():
                    file_path = os.path.join(root, file)
                    records = process_single_pdf(file_path, converter)

                    if records:
                        for r in records:
                            out_file.write(json.dumps(r) + '\n')
                        out_file.flush()
                        found_generators += len(records)

                    processed_count += 1
                    if processed_count % 5 == 0:
                        print(f"--- Status: {processed_count} files checked | {found_generators} generators logged ---")

print(f"\n--- SCRAPING COMPLETE ---")
print(f"Total Generator Records: {found_generators}")

# New Section